In [0]:
# Create catalog and schemas
spark.sql("CREATE CATALOG IF NOT EXISTS education")
spark.sql("CREATE SCHEMA IF NOT EXISTS education.bronze_schema")
spark.sql("CREATE SCHEMA IF NOT EXISTS education.silver_schema")
spark.sql("CREATE SCHEMA IF NOT EXISTS education.gold_schema")

In [0]:
# Load student_info to bronze
student_info_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/databricks-datasets/oulad/studentInfo.csv")
student_info_df.write.format("delta").mode("overwrite").saveAsTable("education.bronze_schema.student_info")

In [0]:
# Load student_registration to bronze
student_registration_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/databricks-datasets/oulad/studentRegistration.csv")
student_registration_df.write.format("delta").mode("overwrite").saveAsTable("education.bronze_schema.student_registration")

In [0]:
# Load assessments to bronze
assessments_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/databricks-datasets/oulad/assessments.csv")
assessments_df.write.format("delta").mode("overwrite").saveAsTable("education.bronze_schema.assessments")

In [0]:
# Load student_assessments to bronze
student_assessments_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/databricks-datasets/oulad/studentAssessment.csv")
student_assessments_df.write.format("delta").mode("overwrite").saveAsTable("education.bronze_schema.student_assessments")

In [0]:
# Load courses to bronze
courses_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/databricks-datasets/oulad/courses.csv")
courses_df.write.format("delta").mode("overwrite").saveAsTable("education.bronze_schema.courses")

In [0]:
# Load vle to bronze
vle_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/databricks-datasets/oulad/vle.csv")
vle_df.write.format("delta").mode("overwrite").saveAsTable("education.bronze_schema.vle")

In [0]:
# Load student_vle to bronze
student_vle_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/databricks-datasets/oulad/studentVle.csv")
student_vle_df.write.format("delta").mode("overwrite").saveAsTable("education.bronze_schema.student_vle")

In [0]:
%sql
SELECT * FROM education.bronze_schema.student_info LIMIT 10

In [0]:
%sql
SELECT COUNT(*) FROM education.bronze_schema.student_info

## Silver Layer Transformations
Clean and transform bronze data into silver tables

In [0]:
# Load bronze tables
student_info = spark.table("education.bronze_schema.student_info")
student_reg = spark.table("education.bronze_schema.student_registration")
assessments = spark.table("education.bronze_schema.assessments")
student_assessments = spark.table("education.bronze_schema.student_assessments")
courses = spark.table("education.bronze_schema.courses")
vle = spark.table("education.bronze_schema.vle")

In [0]:
# Create silver_student_info
silver_student_info = student_info.dropDuplicates()

silver_student_info.write.format("delta").mode("overwrite") \
    .saveAsTable("education.silver_schema.student_info")

In [0]:
# Create silver_student_registration
silver_student_registration = student_reg.dropDuplicates()

silver_student_registration.write.format("delta").mode("overwrite") \
    .saveAsTable("education.silver_schema.student_registration")

In [0]:
# Create silver_assessments
silver_assessments = assessments.dropDuplicates()

silver_assessments.write.format("delta").mode("overwrite") \
    .saveAsTable("education.silver_schema.assessments")

In [0]:
# Create silver_student_assessments
silver_student_assessments = student_assessments.dropDuplicates()

silver_student_assessments.write.format("delta").mode("overwrite") \
    .saveAsTable("education.silver_schema.student_assessments")

In [0]:
# Create silver_courses
silver_courses = courses.dropDuplicates()

silver_courses.write.format("delta").mode("overwrite") \
    .saveAsTable("education.silver_schema.courses")

In [0]:
# Create silver_vle
silver_vle = vle.dropDuplicates()

silver_vle.write.format("delta").mode("overwrite") \
    .saveAsTable("education.silver_schema.vle")

## Create Enriched Student Dataset
Join multiple tables to create comprehensive student view

In [0]:
# Join all student-related tables
silver_student_full = silver_student_info \
    .join(silver_student_registration,
          ["id_student", "code_module", "code_presentation"], "left") \
    .join(silver_student_assessments, "id_student", "left") \
    .join(silver_assessments, "id_assessment", "left") \
    .drop(silver_assessments.code_module) \
    .drop(silver_assessments.code_presentation) \
    .join(silver_courses, ["code_module", "code_presentation"], "left")

In [0]:
from pyspark.sql.functions import col, when

# Add enrichment columns
silver_student_enriched = silver_student_full \
    .withColumn("enrollment_duration", 
                when(col("date_unregistration").isNull(), 
                     col("module_presentation_length") - col("date_registration"))
                .otherwise(col("date_unregistration") - col("date_registration"))) \
    .withColumn("risk_flag",
                when((col("final_result") == "Fail") | (col("final_result") == "Withdrawn"), "High")
                .when(col("num_of_prev_attempts") > 0, "Medium")
                .otherwise("Low")) \
    .withColumn("weighted_score", col("score") * col("weight"))

In [0]:
# Write enriched student data
silver_student_enriched.write.format("delta").mode("overwrite") \
    .saveAsTable("education.silver_schema.student_enriched")

In [0]:
%sql
SELECT risk_flag, COUNT(*) as student_count
FROM education.silver_schema.student_enriched
GROUP BY risk_flag
ORDER BY student_count DESC

In [0]:
%